# Model Comparison — DistilBERT vs RoBERTa

Train both models on the same KCS dataset split and compare accuracy, loss, and inference quality.

| Model | Parameters | Architecture |
|-------|-----------|-------------|
| `distilbert-base-uncased` | 66M | Distilled BERT, faster, lighter |
| `roberta-base` | 125M | Optimized BERT, trained on more data with better hyperparameters |

---
## 0. Install Dependencies

In [112]:
!pip install -q transformers datasets evaluate accelerate boto3 s3fs

21710.25s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


---
## 1. Imports and Configuration

In [113]:
import os
import json
import gc
import numpy as np
import torch
import evaluate
from pathlib import Path
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

MODELS = [
    {
        "name": "distilbert-base-uncased",
        "lr": 2e-5,
        "batch_size": 16,
        "output_dir": "./results-distilbert",
        "save_dir": "./model-distilbert",
    },
    {
        "name": "roberta-base",
        "lr": 1e-5,
        "batch_size": 16,
        "output_dir": "./results-roberta",
        "save_dir": "./model-roberta",
    },
]

NUM_LABELS = 6
EPOCHS = 10
SEED = 42
DATA_FILE = "data/ocp-issues-v2.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


Device: cuda


---
## 2. Label Mapping

In [114]:
label2id = {
    "crashloop_backoff": 0,
    "image_pull_error": 1,
    "route_503": 2,
    "pvc_pending": 3,
    "quota_exceeded": 4,
    "scheduling_constraint": 5,
}
id2label = {v: k for k, v in label2id.items()}

labels_path = Path("data/labels.json")
labels_path.parent.mkdir(parents=True, exist_ok=True)
with open(labels_path, "w") as f:
    json.dump({"label2id": label2id, "id2label": {str(k): v for k, v in id2label.items()}}, f, indent=2)

print(f"Labels: {label2id}")

Labels: {'crashloop_backoff': 0, 'image_pull_error': 1, 'route_503': 2, 'pvc_pending': 3, 'quota_exceeded': 4, 'scheduling_constraint': 5}


---
## 3. Load and Split Dataset

Same split for both models — fair comparison.

In [115]:
raw_dataset = load_dataset("csv", data_files=DATA_FILE)["train"]
split = raw_dataset.train_test_split(test_size=0.2, seed=SEED)

def encode_labels(example):
    example["label"] = label2id[example["label"]]
    return example

train_raw = split["train"].map(encode_labels)
eval_raw = split["test"].map(encode_labels)

print(f"Training:   {len(train_raw)} samples")
print(f"Evaluation: {len(eval_raw)} samples")

train_counts = Counter(train_raw["label"])
print("\nTrain distribution:")
for lid in sorted(train_counts):
    print(f"  {id2label[lid]}: {train_counts[lid]}")

Training:   365 samples
Evaluation: 92 samples

Train distribution:
  crashloop_backoff: 69
  image_pull_error: 53
  route_503: 64
  pvc_pending: 62
  quota_exceeded: 62
  scheduling_constraint: 55


---
## 4. Training Function

Reusable function that tokenizes, trains, evaluates, and returns results for any model.

In [116]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)


def train_and_evaluate(model_cfg, train_data, eval_data):
    """Train a model and return metrics + trainer."""
    name = model_cfg["name"]
    print(f"\n{'='*60}")
    print(f"  TRAINING: {name}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(name)

    def tokenize(examples):
        return tokenizer(examples["text"], truncation=True, max_length=512)

    train_ds = train_data.map(tokenize, batched=True, desc=f"Tokenize train ({name})")
    eval_ds = eval_data.map(tokenize, batched=True, desc=f"Tokenize eval ({name})")

    model = AutoModelForSequenceClassification.from_pretrained(
        name,
        num_labels=NUM_LABELS,
        label2id=label2id,
        id2label=id2label,
    )

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {total_params:,}")

    training_args = TrainingArguments(
        output_dir=model_cfg["output_dir"],
        eval_strategy="epoch",
        save_strategy="no",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=model_cfg["batch_size"],
        per_device_eval_batch_size=model_cfg["batch_size"],
        learning_rate=model_cfg["lr"],
        weight_decay=0.01,
        logging_steps=10,
        report_to="none",
        seed=SEED,
    )

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        compute_metrics=compute_metrics,
        data_collator=collator,
    )

    train_result = trainer.train()

    eval_metrics = [m for m in trainer.state.log_history if "eval_accuracy" in m]
    best_acc = max(m["eval_accuracy"] for m in eval_metrics) if eval_metrics else 0
    best_epoch = next(m["epoch"] for m in eval_metrics if m["eval_accuracy"] == best_acc)
    final_loss = eval_metrics[-1]["eval_loss"] if eval_metrics else 0

    trainer.save_model(model_cfg["save_dir"])
    tokenizer.save_pretrained(model_cfg["save_dir"])

    result = {
        "name": name,
        "params": total_params,
        "best_accuracy": best_acc,
        "best_epoch": best_epoch,
        "final_eval_loss": final_loss,
        "train_loss": train_result.metrics["train_loss"],
        "train_runtime": train_result.metrics["train_runtime"],
        "save_dir": model_cfg["save_dir"],
        "eval_history": eval_metrics,
    }

    print(f"\n  Best accuracy: {best_acc:.4f} (epoch {best_epoch})")
    print(f"  Final eval loss: {final_loss:.4f}")
    print(f"  Training time: {train_result.metrics['train_runtime']:.1f}s")

    del model, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

---
## 5. Train Both Models

In [117]:
results = []
for cfg in MODELS:
    r = train_and_evaluate(cfg, train_raw, eval_raw)
    results.append(r)

print(f"\nAll training complete.")


  TRAINING: distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 66,958,086


Epoch,Training Loss,Validation Loss,Accuracy
1,1.604945,1.415020,0.608696
2,1.002385,0.714589,0.978261
3,0.524240,0.310361,0.978261
4,0.205437,0.145000,1.000000
5,0.132326,0.086199,1.000000
6,0.116800,0.062090,1.000000
7,0.056376,0.052471,1.000000
8,0.051508,0.046442,1.000000
9,0.086186,0.043792,1.000000
10,0.061821,0.042223,1.000000


...

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 124,650,246


Epoch,Training Loss,Validation Loss,Accuracy
1,1.755828,1.666034,0.467391
2,1.173268,0.673122,0.967391
3,0.501713,0.207666,0.978261
4,0.155834,0.083129,1.000000
5,0.099839,0.051571,0.989130
6,0.094159,0.026965,1.000000
7,0.030653,0.020177,1.000000
8,0.025752,0.018280,1.000000
9,0.076256,0.016915,1.000000
10,0.048861,0.016251,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Best accuracy: 1.0000 (epoch 4.0)
  Final eval loss: 0.0163
  Training time: 20.4s

All training complete.


---
## 6. Side-by-Side Comparison

In [118]:
print(f"{'Metric':<25} {'DistilBERT':>15} {'RoBERTa':>15}")
print("-" * 55)

r0, r1 = results[0], results[1]

rows = [
    ("Parameters", f"{r0['params']:,}", f"{r1['params']:,}"),
    ("Best Accuracy", f"{r0['best_accuracy']:.4f}", f"{r1['best_accuracy']:.4f}"),
    ("Best Epoch", f"{r0['best_epoch']:.0f}", f"{r1['best_epoch']:.0f}"),
    ("Final Eval Loss", f"{r0['final_eval_loss']:.4f}", f"{r1['final_eval_loss']:.4f}"),
    ("Train Loss", f"{r0['train_loss']:.4f}", f"{r1['train_loss']:.4f}"),
    ("Training Time (s)", f"{r0['train_runtime']:.1f}", f"{r1['train_runtime']:.1f}"),
]

for label, v0, v1 in rows:
    marker = ""
    if label == "Best Accuracy":
        marker = " <--" if r0["best_accuracy"] > r1["best_accuracy"] else (" <--" if r1["best_accuracy"] > r0["best_accuracy"] else "")
        if r1["best_accuracy"] > r0["best_accuracy"]:
            print(f"{label:<25} {v0:>15} {v1:>15}{marker}")
        elif r0["best_accuracy"] > r1["best_accuracy"]:
            print(f"{label:<25} {v0:>15}{marker} {v1:>15}")
        else:
            print(f"{label:<25} {v0:>15} {v1:>15}  (tie)")
    else:
        print(f"{label:<25} {v0:>15} {v1:>15}")

winner = r0 if r0["best_accuracy"] >= r1["best_accuracy"] else r1
if r0["best_accuracy"] == r1["best_accuracy"]:
    winner = r0 if r0["final_eval_loss"] < r1["final_eval_loss"] else r1

print(f"\nWinner: {winner['name']} (accuracy={winner['best_accuracy']:.4f}, loss={winner['final_eval_loss']:.4f})")

Metric                         DistilBERT         RoBERTa
-------------------------------------------------------
Parameters                     66,958,086     124,650,246
Best Accuracy                      1.0000          1.0000  (tie)
Best Epoch                              4               4
Final Eval Loss                    0.0422          0.0163
Train Loss                         0.3907          0.3984
Training Time (s)                    11.0            20.4

Winner: roberta-base (accuracy=1.0000, loss=0.0163)


---
## 7. Accuracy Over Epochs

In [119]:
print(f"{'Epoch':>6}  {'DistilBERT Acc':>15} {'DistilBERT Loss':>16}  {'RoBERTa Acc':>12} {'RoBERTa Loss':>13}")
print("-" * 70)

for i in range(len(results[0]["eval_history"])):
    e0 = results[0]["eval_history"][i]
    e1 = results[1]["eval_history"][i]
    print(
        f"{e0['epoch']:>6.0f}"
        f"  {e0['eval_accuracy']:>15.4f} {e0['eval_loss']:>16.4f}"
        f"  {e1['eval_accuracy']:>12.4f} {e1['eval_loss']:>13.4f}"
    )

 Epoch   DistilBERT Acc  DistilBERT Loss   RoBERTa Acc  RoBERTa Loss
----------------------------------------------------------------------
     1           0.6087           1.4150        0.4674        1.6660
     2           0.9783           0.7146        0.9674        0.6731
     3           0.9783           0.3104        0.9783        0.2077
     4           1.0000           0.1450        1.0000        0.0831
     5           1.0000           0.0862        0.9891        0.0516
     6           1.0000           0.0621        1.0000        0.0270
     7           1.0000           0.0525        1.0000        0.0202
     8           1.0000           0.0464        1.0000        0.0183
     9           1.0000           0.0438        1.0000        0.0169
    10           1.0000           0.0422        1.0000        0.0163


---
## 8. Inference Comparison

Test both models on the same examples and compare predictions.

In [122]:
test_cases = [
    ("crashloop_backoff", "Deployment: api-server; Pod state: CrashLoopBackOff; Restarts: 15; Events: Back-off restarting failed container; Logs: FATAL: password authentication failed for user postgres"),
    ("image_pull_error", "Deployment: web-frontend; Pod state: ImagePullBackOff; Events: Failed to pull image registry.internal/frontend:v3; unauthorized: authentication required"),
    ("route_503", "Route: payment-route; HTTP 503; Service endpoints: 0; Pods ready: 0/2; Deployment replicas: 2 desired 0 available"),
    ("pvc_pending", "PVC: data-volume; Status: Pending; Events: storageclass fast-ssd not found; No default StorageClass configured"),
    ("quota_exceeded", "Deployment: ml-worker; Pod state: Pending; Events: exceeded quota: gpu-quota; requested: nvidia.com/gpu=2; used: 4; limited: 4"),
    ("scheduling_constraint", "Deployment: cache-server; Pod state: Pending; Events: 0/6 nodes are available: 3 node(s) had taint dedicated=gpu:NoSchedule that the pod didn't tolerate; 3 had master taint"),
]

model_predictions = {}
for r in results:
    tok = AutoTokenizer.from_pretrained(r["save_dir"])
    mdl = AutoModelForSequenceClassification.from_pretrained(r["save_dir"])
    mdl.eval()

    preds = []
    for expected, text in test_cases:
        inputs = tok(text, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            outputs = mdl(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)[0]
        pred_id = torch.argmax(probs).item()
        preds.append((id2label[pred_id], probs[pred_id].item()))

    model_predictions[r["name"]] = preds
    del mdl
    gc.collect()

print(f"{'Expected':<25} {'DistilBERT':<30} {'RoBERTa':<30}")
print("-" * 85)

for i, (expected, text) in enumerate(test_cases):
    d_label, d_conf = model_predictions[results[0]["name"]][i]
    r_label, r_conf = model_predictions[results[1]["name"]][i]
    d_check = "ok" if d_label == expected else "WRONG"
    r_check = "ok" if r_label == expected else "WRONG"
    print(f"{expected:<25} {d_label} ({d_conf:.1%}) [{d_check}]{'':>5} {r_label} ({r_conf:.1%}) [{r_check}]")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Expected                  DistilBERT                     RoBERTa                       
-------------------------------------------------------------------------------------
crashloop_backoff         crashloop_backoff (97.3%) [ok]      crashloop_backoff (99.0%) [ok]
image_pull_error          image_pull_error (96.4%) [ok]      image_pull_error (98.7%) [ok]
route_503                 route_503 (97.4%) [ok]      route_503 (98.5%) [ok]
pvc_pending               pvc_pending (96.5%) [ok]      pvc_pending (98.9%) [ok]
quota_exceeded            quota_exceeded (97.3%) [ok]      quota_exceeded (98.8%) [ok]
scheduling_constraint     scheduling_constraint (94.9%) [ok]      scheduling_constraint (98.4%) [ok]


---
## 9. Upload Winning Model to MinIO

In [121]:
import boto3
from datetime import datetime, timezone
from botocore.client import Config
import shutil

BUCKET_NAME = os.environ.get("AWS_S3_BUCKET", "fine-tuning")
S3_PREFIX = "kcs-classifier/v2"


def build_s3_client():
    return boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        config=Config(signature_version="s3v4"),
        region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
    )


def upload_directory_to_s3(local_dir, bucket, prefix):
    s3 = build_s3_client()
    local_dir = Path(local_dir)
    for file_path in local_dir.rglob("*"):
        if file_path.is_file():
            s3_key = f"{prefix}/{file_path.relative_to(local_dir).as_posix()}"
            print(f"  {file_path.name} -> s3://{bucket}/{s3_key}")
            s3.upload_file(str(file_path), bucket, s3_key)


save_dir = Path(winner["save_dir"])
shutil.copy("data/labels.json", save_dir / "labels.json")

comparison_meta = {
    "winner": winner["name"],
    "comparison": [
        {"model": r["name"], "accuracy": r["best_accuracy"], "eval_loss": r["final_eval_loss"], "params": r["params"]}
        for r in results
    ],
    "dataset": DATA_FILE,
    "epochs": EPOCHS,
    "exported_at_utc": datetime.now(timezone.utc).isoformat(),
}
with open(save_dir / "metadata.json", "w") as f:
    json.dump(comparison_meta, f, indent=2)

print(f"Uploading winner ({winner['name']}) to s3://{BUCKET_NAME}/{S3_PREFIX}")
upload_directory_to_s3(save_dir, BUCKET_NAME, S3_PREFIX)
print("\nUpload complete.")

Uploading winner (roberta-base) to s3://fine-tuning/kcs-classifier/v2
  training_args.bin -> s3://fine-tuning/kcs-classifier/v2/training_args.bin
  labels.json -> s3://fine-tuning/kcs-classifier/v2/labels.json
  metadata.json -> s3://fine-tuning/kcs-classifier/v2/metadata.json
  model.safetensors -> s3://fine-tuning/kcs-classifier/v2/model.safetensors
  tokenizer_config.json -> s3://fine-tuning/kcs-classifier/v2/tokenizer_config.json
  config.json -> s3://fine-tuning/kcs-classifier/v2/config.json
  tokenizer.json -> s3://fine-tuning/kcs-classifier/v2/tokenizer.json

Upload complete.


---
## Summary

This notebook trained and compared two models on the same dataset split:

- **DistilBERT** — lighter, faster, good baseline
- **RoBERTa** — heavier, typically more accurate for NLP classification

The winning model was uploaded to MinIO at `kcs-classifier/v2`.